CANINE + CONTRASTIVE LEARNING + PROTOTYPICAL - 150 classes



In [1]:
!pip install transformers torch nlpaug -q

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import json, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CanineTokenizer, CanineModel, get_linear_schedule_with_warmup
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
CLINC_PATH    = "/content/drive/MyDrive/NLP/oos-eval/data/data_full.json"
NOISY_PATH    = "/content/drive/MyDrive/NLP/bert-clinc/noisy_test_set2.json"
SAVE_DIR      = "/content/drive/MyDrive/NLP/bert-clinc/canine_model"
MAX_LENGTH    = 1024

import os; os.makedirs(SAVE_DIR, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 11.4 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda


In [2]:
with open(CLINC_PATH) as f:
    data = json.load(f)

train_texts  = [x[0] for x in data["train"]]
train_labels = [x[1] for x in data["train"]]
test_texts   = [x[0] for x in data["test"]]
test_labels  = [x[1] for x in data["test"]]
val_texts    = [x[0] for x in data["val"]]
val_labels   = [x[1] for x in data["val"]]

all_labels = sorted(set(train_labels))
label2id   = {l: i for i, l in enumerate(all_labels)}
num_labels = len(all_labels)
train_ids  = [label2id[l] for l in train_labels]
test_ids   = [label2id[l] for l in test_labels]
val_ids    = [label2id[l] for l in val_labels]

with open(NOISY_PATH) as f:
    noisy_data = json.load(f)

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
print(f"Train: {len(train_texts)} | Test: {len(test_texts)} | Intents: {num_labels}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/854 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

Train: 15000 | Test: 4500 | Intents: 150


In [3]:
class CanineClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.encoder    = CanineModel.from_pretrained("google/canine-s")
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, num_labels)

    def get_embedding(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])

    def forward(self, input_ids, attention_mask, labels=None):
        emb    = self.get_embedding(input_ids, attention_mask)
        logits = self.classifier(emb)
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return loss, logits
        return logits

class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts    = texts
        self.labels   = labels
        self.tok      = tokenizer
        self.maxlen   = max_length

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True,
                       padding="max_length", max_length=self.maxlen,
                       return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx])
        }

def evaluate_standard(model, texts, label_ids, batch_size=32):
    model.eval()
    preds, labels_all = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LENGTH, return_tensors="pt")
        with torch.no_grad():
            emb    = model.get_embedding(enc["input_ids"].to(device),
                                         enc["attention_mask"].to(device))
            logits = model.classifier(emb)
            pred   = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend(pred)
        labels_all.extend(label_ids[i:i+batch_size])
    return accuracy_score(labels_all, preds)

print("Model and dataset classes defined")

Model and dataset classes defined


In [4]:
model = CanineClassifier(num_labels=num_labels).to(device)

train_dataset = SimpleDataset(train_texts, train_ids, tokenizer, MAX_LENGTH)
val_dataset   = SimpleDataset(val_texts, val_ids, tokenizer, MAX_LENGTH)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=32)

EPOCHS    = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS)

print("Starting clean training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        loss, _ = model(ids, mask, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(train_loader)} Loss: {loss.item():.4f}")

    val_acc = evaluate_standard(model, val_texts, val_ids)
    print(f"Epoch {epoch+1} done — Avg Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}%")

    # Save after every epoch
    torch.save(model.state_dict(), f"{SAVE_DIR}/canine_clean_epoch{epoch+1}.pt")
    print(f"Saved checkpoint epoch {epoch+1}")

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/528M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting clean training...
Epoch 1 Step 0/1875 Loss: 5.0005
Epoch 1 Step 200/1875 Loss: 4.9900
Epoch 1 Step 400/1875 Loss: 5.0278
Epoch 1 Step 600/1875 Loss: 4.9537
Epoch 1 Step 800/1875 Loss: 4.8989
Epoch 1 Step 1000/1875 Loss: 4.9404
Epoch 1 Step 1200/1875 Loss: 4.7509
Epoch 1 Step 1400/1875 Loss: 4.5986
Epoch 1 Step 1600/1875 Loss: 4.2729
Epoch 1 Step 1800/1875 Loss: 4.3956
Epoch 1 done — Avg Loss: 4.8353 | Val Acc: 17.87%
Saved checkpoint epoch 1
Epoch 2 Step 0/1875 Loss: 4.3537
Epoch 2 Step 200/1875 Loss: 2.8074
Epoch 2 Step 400/1875 Loss: 3.3546
Epoch 2 Step 600/1875 Loss: 1.6411
Epoch 2 Step 800/1875 Loss: 2.1477
Epoch 2 Step 1000/1875 Loss: 2.3171
Epoch 2 Step 1200/1875 Loss: 1.5475
Epoch 2 Step 1400/1875 Loss: 1.5009
Epoch 2 Step 1600/1875 Loss: 0.7143
Epoch 2 Step 1800/1875 Loss: 0.0770
Epoch 2 done — Avg Loss: 1.7027 | Val Acc: 82.93%
Saved checkpoint epoch 2
Epoch 3 Step 0/1875 Loss: 0.2044
Epoch 3 Step 200/1875 Loss: 0.5086
Epoch 3 Step 400/1875 Loss: 1.1738
Epoch 3 Step 6

In [5]:

print(f"{'Noise Type':<20} {'Accuracy':>10}")
print("-" * 32)
noise_variants = {
    "original":      test_texts,
    "casing":        noisy_data["casing"],
    "keyboard":      noisy_data["keyboard"],
    "spelling":      noisy_data["spelling"],
    "synonyms":      noisy_data["synonyms"],
    "abbreviations": noisy_data["abbreviations"],
    "whisper":       noisy_data["whisper"],
}
for noise_type, texts in noise_variants.items():
    acc = evaluate_standard(model, texts, test_ids[:len(texts)])
    print(f"{noise_type:<20} {acc*100:>9.2f}%")

Noise Type             Accuracy
--------------------------------
original                 91.60%
casing                   82.24%
keyboard                 69.76%
spelling                 87.22%
synonyms                 91.60%
abbreviations            89.82%
whisper                  89.47%


In [6]:


class ContrastiveDataset(Dataset):
    def __init__(self, clean_texts, noisy_texts, labels, tokenizer, max_length):
        self.clean  = clean_texts
        self.noisy  = noisy_texts
        self.labels = labels
        self.tok    = tokenizer
        self.maxlen = max_length

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        def enc(t):
            return self.tok(t, truncation=True, padding="max_length",
                            max_length=self.maxlen, return_tensors="pt")
        c = enc(self.clean[idx])
        n = enc(self.noisy[idx])
        return {
            "clean_input_ids":      c["input_ids"].squeeze(),
            "clean_attention_mask": c["attention_mask"].squeeze(),
            "noisy_input_ids":      n["input_ids"].squeeze(),
            "noisy_attention_mask": n["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx])
        }

kb_aug = nac.KeyboardAug(aug_char_p=0.15)
sp_aug = naw.SpellingAug(aug_p=0.2)

def generate_noisy(text):
    choice = random.choices(["keyboard", "spelling"], weights=[0.7, 0.3])[0]
    try:
        return kb_aug.augment(text)[0] if choice == "keyboard" else sp_aug.augment(text)[0]
    except:
        return text

print("Generating noisy pairs...")
noisy_train_texts = [generate_noisy(t) for t in train_texts]

cont_dataset = ContrastiveDataset(train_texts, noisy_train_texts,
                                   train_ids, tokenizer, MAX_LENGTH)
cont_loader  = DataLoader(cont_dataset, batch_size=8, shuffle=True)

CONT_EPOCHS = 3
optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=len(cont_loader),
    num_training_steps=len(cont_loader) * CONT_EPOCHS)

print("Starting contrastive fine-tuning...")
for epoch in range(CONT_EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(cont_loader):
        c_ids  = batch["clean_input_ids"].to(device)
        c_mask = batch["clean_attention_mask"].to(device)
        n_ids  = batch["noisy_input_ids"].to(device)
        n_mask = batch["noisy_attention_mask"].to(device)
        labels = batch["label"].to(device)

        clean_emb = model.get_embedding(c_ids, c_mask)
        noisy_emb = model.get_embedding(n_ids, n_mask)
        logits    = model.classifier(clean_emb)

        ce_loss = nn.CrossEntropyLoss()(logits, labels)

        cn  = F.normalize(clean_emb, dim=-1)
        nn_ = F.normalize(noisy_emb, dim=-1)
        B   = cn.size(0)
        embs = torch.cat([cn, nn_], dim=0)
        sim  = torch.matmul(embs, embs.T) / 0.07
        mask = torch.eye(2*B, device=embs.device).bool()
        sim.masked_fill_(mask, float('-inf'))
        cl_labels = torch.cat([torch.arange(B, 2*B), torch.arange(B)]).to(embs.device)
        cl_loss   = nn.CrossEntropyLoss()(sim, cl_labels)

        loss = ce_loss + 0.3 * cl_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if step % 200 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(cont_loader)} "
                  f"Loss: {loss.item():.4f} CE: {ce_loss.item():.4f} CL: {cl_loss.item():.4f}")

    print(f"Epoch {epoch+1} done — Avg Loss: {total_loss/len(cont_loader):.4f}")

    # Save after every epoch
    torch.save(model.state_dict(), f"{SAVE_DIR}/canine_contrastive_epoch{epoch+1}.pt")
    print(f"Saved contrastive checkpoint epoch {epoch+1}")

Generating noisy pairs...
Starting contrastive fine-tuning...
Epoch 1 Step 0/1875 Loss: 0.0692 CE: 0.0091 CL: 0.2006
Epoch 1 Step 200/1875 Loss: 0.0480 CE: 0.0090 CL: 0.1299
Epoch 1 Step 400/1875 Loss: 0.1193 CE: 0.0188 CL: 0.3348
Epoch 1 Step 600/1875 Loss: 0.0554 CE: 0.0306 CL: 0.0826
Epoch 1 Step 800/1875 Loss: 0.0279 CE: 0.0026 CL: 0.0841
Epoch 1 Step 1000/1875 Loss: 0.4376 CE: 0.4311 CL: 0.0220
Epoch 1 Step 1200/1875 Loss: 0.5585 CE: 0.5368 CL: 0.0723
Epoch 1 Step 1400/1875 Loss: 0.5461 CE: 0.5298 CL: 0.0543
Epoch 1 Step 1600/1875 Loss: 0.2894 CE: 0.2806 CL: 0.0293
Epoch 1 Step 1800/1875 Loss: 0.7923 CE: 0.7899 CL: 0.0080
Epoch 1 done — Avg Loss: 0.1593
Saved contrastive checkpoint epoch 1
Epoch 2 Step 0/1875 Loss: 0.0996 CE: 0.0702 CL: 0.0980
Epoch 2 Step 200/1875 Loss: 0.1147 CE: 0.1032 CL: 0.0382
Epoch 2 Step 400/1875 Loss: 0.0219 CE: 0.0120 CL: 0.0330
Epoch 2 Step 600/1875 Loss: 0.0069 CE: 0.0016 CL: 0.0178
Epoch 2 Step 800/1875 Loss: 0.3380 CE: 0.3315 CL: 0.0216
Epoch 2 Step 

In [7]:

print("Building prototypes...")
model.eval()
embs_per_class = {i: [] for i in range(num_labels)}

for i, (text, label) in enumerate(zip(train_texts, train_ids)):
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_embedding(enc["input_ids"].to(device),
                                   enc["attention_mask"].to(device))
    embs_per_class[label].append(emb.squeeze().cpu())
    if i % 1000 == 0:
        print(f"  {i}/{len(train_texts)}")

prototypes = torch.stack([
    torch.stack(embs_per_class[i]).mean(dim=0)
    for i in range(num_labels)
]).to(device)

torch.save(prototypes, f"{SAVE_DIR}/prototypes.pt")
print("Prototypes saved")

# ── Evaluate with prototypical classifier ─────────────────
def evaluate_proto(texts, label_ids):
    model.eval()
    preds, labels_all = [], []
    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]
        enc   = tokenizer(batch, truncation=True, padding=True,
                          max_length=MAX_LENGTH, return_tensors="pt")
        with torch.no_grad():
            emb  = model.get_embedding(enc["input_ids"].to(device),
                                        enc["attention_mask"].to(device))
            dist = torch.cdist(emb, prototypes)
            pred = torch.argmin(dist, dim=-1).cpu().numpy()
        preds.extend(pred)
        labels_all.extend(label_ids[i:i+32])
    return accuracy_score(labels_all, preds)

print(f"\n{'Noise Type':<20} {'Accuracy':>10}")
print("-" * 32)
for noise_type, texts in noise_variants.items():
    acc = evaluate_proto(texts, test_ids[:len(texts)])
    print(f"{noise_type:<20} {acc*100:>9.2f}%")

Building prototypes...
  0/15000
  1000/15000
  2000/15000
  3000/15000
  4000/15000
  5000/15000
  6000/15000
  7000/15000
  8000/15000
  9000/15000
  10000/15000
  11000/15000
  12000/15000
  13000/15000
  14000/15000
Prototypes saved

Noise Type             Accuracy
--------------------------------
original                 92.02%
casing                   77.33%
keyboard                 83.82%
spelling                 89.04%
synonyms                 92.02%
abbreviations            90.82%
whisper                  89.76%
